# LendInsight — Notebook 1: Data Exploration
**Project**: LendInsight Lending Business Decision Support System  
**Purpose**: Understand the raw structure, distributions and quality of the cleaned dataset before analysis.  
**Dataset**: `credit_risk_cleaned.csv` — 32,411 loan records, 15 columns


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# Plot style
sns.set_theme(style="whitegrid", palette="Blues_d")
plt.rcParams.update({"figure.dpi": 120, "font.family": "sans-serif"})

CLEAN_PATH = r"C:\data_analyst\LendInsight\01_data\clean\credit_risk_cleaned.csv"
df = pd.read_csv(CLEAN_PATH)
print(f"Dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")


## 1. Dataset Shape & Column Overview

In [ ]:

print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]}")
print()
print(df.dtypes.to_string())


## 2. First 5 Rows

In [ ]:
df.head()

## 3. Descriptive Statistics — Numeric Columns

In [ ]:
df.describe().round(2)

## 4. Null Value Check (Post-ETL)

In [ ]:

nulls = df.isnull().sum()
print("Null counts per column:")
print(nulls[nulls > 0] if nulls.sum() > 0 else "[OK] Zero nulls — dataset is clean")


## 5. Target Variable Distribution — default_flag

In [ ]:

counts = df["default_flag"].value_counts()
labels = ["Repaid (0)", "Defaulted (1)"]
colors = ["#2ECC71", "#E74C3C"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count bar
axes[0].bar(labels, counts.values, color=colors, edgecolor="white", width=0.5)
axes[0].set_title("Loan Outcome — Count", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Number of Loans")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 200, f"{v:,}", ha="center", fontweight="bold")

# Pie
axes[1].pie(counts.values, labels=labels, colors=colors, autopct="%1.1f%%",
            startangle=90, wedgeprops=dict(edgecolor="white", linewidth=2))
axes[1].set_title("Loan Outcome — Share", fontsize=13, fontweight="bold")

plt.suptitle("Default vs Repaid Distribution", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(r"C:\data_analyst\LendInsight\03_python\chart_01_target_distribution.png",
            bbox_inches="tight")
plt.show()
print(f"Default Rate: {counts[1]/counts.sum()*100:.2f}%")


## 6. Loan Grade Distribution

In [ ]:

grade_counts = df["loan_grade"].value_counts().sort_index()
colors_grade = ["#1A5276","#2E86C1","#5DADE2","#F39C12","#E67E22","#C0392B","#7B241C"]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(grade_counts.index, grade_counts.values, color=colors_grade, edgecolor="white")
ax.set_title("Loan Volume by Grade (A = Lowest Risk → G = Highest Risk)",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Loan Grade"); ax.set_ylabel("Number of Loans")
for bar, val in zip(bars, grade_counts.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+100,
            f"{val:,}", ha="center", fontsize=9, fontweight="bold")
plt.tight_layout()
plt.savefig(r"C:\data_analyst\LendInsight\03_python\chart_02_grade_distribution.png",
            bbox_inches="tight")
plt.show()


## 7. Loan Intent Distribution

In [ ]:

intent_counts = df["loan_intent"].value_counts()

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(intent_counts.index, intent_counts.values,
               color=sns.color_palette("Blues_d", len(intent_counts)), edgecolor="white")
ax.set_title("Loan Volume by Intent (Purpose)", fontsize=13, fontweight="bold")
ax.set_xlabel("Number of Loans")
for bar, val in zip(bars, intent_counts.values):
    ax.text(val + 80, bar.get_y()+bar.get_height()/2,
            f"{val:,}", va="center", fontsize=9, fontweight="bold")
plt.tight_layout()
plt.savefig(r"C:\data_analyst\LendInsight\03_python\chart_03_intent_distribution.png",
            bbox_inches="tight")
plt.show()


## 8. Loan Amount Distribution

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df["loan_amnt"], bins=40, color="#2E86C1", edgecolor="white")
axes[0].set_title("Loan Amount Distribution", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Loan Amount (USD)"); axes[0].set_ylabel("Frequency")

sns.boxplot(x="default_flag", y="loan_amnt", data=df, ax=axes[1],
            palette={0:"#2ECC71", 1:"#E74C3C"})
axes[1].set_title("Loan Amount by Outcome", fontsize=13, fontweight="bold")
axes[1].set_xlabel("0 = Repaid | 1 = Defaulted"); axes[1].set_ylabel("Loan Amount (USD)")
axes[1].set_xticklabels(["Repaid","Defaulted"])

plt.tight_layout()
plt.savefig(r"C:\data_analyst\LendInsight\03_python\chart_04_loan_amount.png",
            bbox_inches="tight")
plt.show()
print(f"Avg loan (Repaid)   : ${df[df.default_flag==0].loan_amnt.mean():,.0f}")
print(f"Avg loan (Defaulted): ${df[df.default_flag==1].loan_amnt.mean():,.0f}")


## 9. Key Summary Statistics

In [ ]:

total   = len(df)
defs    = df["default_flag"].sum()
summary = {
    "Total Loans"          : f"{total:,}",
    "Total Defaults"       : f"{defs:,}",
    "Default Rate"         : f"{defs/total*100:.2f}%",
    "Avg Loan Amount"      : f"${df['loan_amnt'].mean():,.2f}",
    "Avg Interest Rate"    : f"{df['loan_int_rate'].mean():.2f}%",
    "Avg DTI (loan/income)": f"{df['loan_percent_income'].mean():.3f}",
    "HIGH Risk Loans"      : f"{(df['risk_category']=='HIGH').sum():,}",
    "LOW Risk Loans"       : f"{(df['risk_category']=='LOW').sum():,}",
}
for k, v in summary.items():
    print(f"  {k:<28}: {v}")
